In [ ]:
import os
import sys
import yaml
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from model.model_manager import ARCEMEPipeline

HERE_DIR = Path.cwd() 
ROOT_DIR = HERE_DIR.parent.parent
print(f"Root directory: {ROOT_DIR}")

In [ ]:
config_path = os.path.join(ROOT_DIR, "model", "config.yaml")

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

# ⚠️ OVERRIDES FÜR DAS NOTEBOOK ⚠️
# Wir wollen im Notebook nur testen, ob der Code durchläuft, nicht wirklich konvergieren!

# 1. Epochen und Folds reduzieren
cfg["training"]["max_epochs"] = 1
cfg["cross_validation"]["k_folds"] = 3 # Wir testen nur den ersten Fold
cfg["data"]["train_data_dir"] = "/scratch/sloeblein/postprocessed"  # kleiner datensatz
cfg["experiment_name"] = "Testing_ConvLSTM_Notebook"

# 2. DataLoader sicherer machen (Jupyter mag Multiprocessing oft nicht)
cfg["data"]["data_loader"]["num_workers"] = 0 
cfg["training"]["batch_size"] = 2 # Klein halten wegen GPU RAM im Notebook

# 3. Kriterien kurz lockern, damit der Dataloader SOFORT einen Patch findet
cfg["data"]["quality_check"]["ctx_min_valid_ratio"] = 0.01
cfg["data"]["quality_check"]["tgt_min_valid_ratio"] = 0.01
cfg["data"]["quality_check"]["ctx_required_fraction"] = 0.1
cfg["data"]["quality_check"]["tgt_min_timesteps"] = 1

print("✅ Config geladen und Notebook-Overrides angewendet!")

In [ ]:
# --- ZELLE 3: Init & Data Preparation ---
# Wir starten im Trainings-Modus
pipeline = ARCEMEPipeline(config=cfg, mode="train", run_dir="/net/home/sloeblein/ARCEME-Vegetation-Recovery/model/wand_db_logs/test_run_dir")

print(f"\nGenerierter Run Directory Pfad: {pipeline.run_dir}")

# Teste die prepare_data Funktion (erstellt cv_splits.json)
all_folds = pipeline.prepare_data()

train_files, val_files = all_folds[0]
print(f"\nFold 0 generiert: {len(train_files)} Train Cubes, {len(val_files)} Val Cubes")

In [ ]:
# --- ZELLE 4: Dataloader Test & Shape Check ---
train_loader, val_loader = pipeline.get_dataloaders(train_files, val_files[:1], fold_idx=0)

# Einen einzelnen Batch aus dem Dataloader ziehen
batch = next(iter(train_loader))
x_context, x_future_feat, y_target, target_mask, meta, baseline_sample = batch

print("📦 TENSOR SHAPES:")
print(f"Context Input:   {x_context.shape} | Erwartet: (Batch, Time, Channels, H, W)")
print(f"Future Features: {x_future_feat.shape} | Erwartet: (Batch, Time, Future_Channels, H, W)")
print(f"Target:          {y_target.shape}")
print(f"Target Mask:     {target_mask.shape}")
print(f"Baseline:        {baseline_sample.shape}")

# Teste, ob der kNDVI Placeholder in x_future_feat WIRKLICH aus Nullen besteht
kndvi_placeholder = x_future_feat[:, :, 0, :, :]
is_empty = torch.all(kndvi_placeholder == 0.0).item()
print(f"\nIst der kNDVI Placeholder korrekt genullt? {'✅ JA' if is_empty else '❌ NEIN'}")
# Prüfe: überall dort, wo y_target == 0.0 ist, muss target_mask == 0 sein
mask_consistent = torch.all(target_mask[y_target == 0.0] == 0).item()
print(f"Sind alle target_mask-Werte bei y_target==0.0 gleich 0? {'✅ JA' if mask_consistent else '❌ NEIN'}")

if not is_empty:
    print(f"WARNUNG: Max Value im Placeholder: {kndvi_placeholder.max().item()}")

In [ ]:
# --- ZELLE 5: Teste den Trainings-Loop ---
print("🚀 Starte Test-Training (1 Fold, 1 Epoche)...")

# Dies ruft intern trainer.fit() auf
# (Durch unsere Overrides in Zelle 2 stoppt das sehr schnell wieder)
pipeline.run_cv(start_fold=0)

print("✅ Trainings-Test abgeschlossen!")

In [ ]:
# --- ZELLE 6: Checkpoint Paths testen ---
# pipeline.run_dir = "/net/home/sloeblein/ARCEME-Vegetation-Recovery/model/wand_db_logs/run_2026-04-14_19-57-17"
run_dir = pipeline.run_dir
print(run_dir)

last_ckpt = pipeline.get_checkpoint_path(fold_idx=0, type="last")
best_ckpt = pipeline.get_checkpoint_path(fold_idx=0, type="best")
overall_best = pipeline.get_best_overall_checkpoint()

print("🔍 GEFUNDENE CHECKPOINTS:")
print(f"Last Checkpoint (Fold 0): {last_ckpt}")
print(f"Best Checkpoint (Fold 0): {best_ckpt}")
print(f"Overall Best:             {overall_best}")

# Sicherstellen, dass die Dateien auch physisch existieren
assert os.path.exists(last_ckpt), "Last Checkpoint Datei fehlt!"
assert os.path.exists(best_ckpt), "Best Checkpoint Datei fehlt!"
print("\n✅ Alle Checkpoints wurden erfolgreich auf der Festplatte gefunden!")

In [ ]:
# --- ZELLE 7: Evaluation Test ---
# 1. Wir erstellen eine Dummy-Textdatei mit 2 Pfaden aus den Validierungsdaten
dummy_test_list = os.path.join(ROOT_DIR, "test_list_dummy.txt")

with open(dummy_test_list, "w") as f:
    f.write(f"{val_files[0]}\n")
    if len(val_files) > 1:
        f.write(f"{val_files[1]}\n")
        
print(f"📄 Dummy Test-List erstellt unter: {dummy_test_list}")

# 2. Pipeline im Eval-Modus initialisieren (nutzt das run_dir von oben)
eval_pipeline = ARCEMEPipeline(config=cfg, mode="eval", run_dir=run_dir)

# 3. Zarr-Pfade auslesen (wie in evaluate.py)
with open(dummy_test_list, 'r') as f:
    custom_test_files = [line.strip() for line in f.readlines() if line.strip().endswith('.zarr')]

print(f"Führe Evaluation auf {len(custom_test_files)} Cubes durch...")

# 4. Evaluation starten (nutzt den best_ckpt aus Zelle 6)
results = eval_pipeline.evaluate(ckpt_path=best_ckpt, test_files=custom_test_files)

print("\n📊 EVALUATION RESULTS:")
print(results)

In [ ]:
# --- ZELLE: Manuelles Validierungs-Debugging im Notebook ---

model = pipeline.load_model(ckpt_path=best_ckpt)
model.eval() # Wichtig: Dropout & BatchNorm auf Eval setzen!

print("\n✅ Model geladen und auf Eval gesetzt!")

# 2. Dataloader holen (z.B. den Val-Loader aus Fold 0)
# train_files, val_files = pipeline.prepare_data()[0]
# _, val_loader = pipeline.get_dataloaders(train_files, val_files, fold_idx=0)

# 3. Einen einzigen Batch manuell laden
batch = next(iter(val_loader))

# --- DER FIX: Batch auf das richtige Device schieben ---
# Wir holen uns das Device, auf dem das Modell aktuell liegt (meist 'cuda:0')
device = next(model.parameters()).device 
print(f"Modell liegt auf Device: {device}")

# Wir entpacken den Batch (Tuple aus 6 Elementen in deinem Fall) 
# und schieben jeden Tensor auf dieses Device, ignorieren aber dicts/lists (wie 'meta').
batch_on_device = []
for item in batch:
    if isinstance(item, torch.Tensor):
        batch_on_device.append(item.to(device))
    else:
        batch_on_device.append(item) # 'meta' (dict) bleibt wie es ist

# Aus der Liste wieder ein Tuple machen (wie es validation_step erwartet)
batch = tuple(batch_on_device)
print("✅ Batch erfolgreich auf das richtige Device verschoben!")
# --------------------------------------------------------

# 4. Den Validation Step aufrufen!
# Das Model legt jetzt alle Resultate in self.validation_step_outputs ab
with torch.no_grad(): # Gradienten-Berechnung ausschalten
    model.validation_step(batch, batch_idx=0)

# 5. Auf die Outputs zugreifen und inspizieren
outputs = model.validation_step_outputs
print(f"Es wurden {len(outputs)} Patches im validation_step_outputs gespeichert.")

# Schaue dir den allerersten Patch an:
first_patch = outputs[0]
print("Keys im Output:", first_patch.keys())
print("Cube ID:", first_patch["cube_id"])
print("Position:", first_patch["top"], first_patch["left"])
print("Shape Prediction:", first_patch["y_pred"].shape)

# # Optional: Plotten!
# import matplotlib.pyplot as plt
# plt.imshow(first_patch["y_pred"][0, 0, :, :].numpy(), cmap="viridis")
# plt.title("Prediction für t=0")
# plt.show()

# # 6. WICHTIG: Speicher wieder leeren, bevor du das nächste Mal testest!
# model.validation_step_outputs.clear()

In [ ]:
import torch
# from getattr import getattr
from tqdm.notebook import tqdm

# 2. Schleife über ALLE Batches im Dataloader
print(f"🚀 Starte Forward Pass über {len(val_loader)} Batches...")

with torch.no_grad(): # Wichtig!
    for batch_idx, batch in enumerate(tqdm(val_loader, desc="Validating")):
        
        # Batch auf GPU schieben
        batch_on_device = []
        for item in batch:
            if isinstance(item, torch.Tensor):
                batch_on_device.append(item.to(device))
            else:
                batch_on_device.append(item)
                
        batch = tuple(batch_on_device)
        
        # Modell füttern -> Speichert Ergebnisse in self.validation_step_outputs
        model.validation_step(batch, batch_idx)

# 3. Jetzt haben wir alle Puzzleteile!
patches_count = len(model.validation_step_outputs)
# For patch_size 128 -> 64 Patches per  cube
# patches_count = num_cubes * 64 / batch_size
# 3 cubes * 64 patches per cube / batch_size 2 = 96 Patches
print(f"\n🧩 Alle Patches gesammelt! Insgesamt: {patches_count} Patches im Speicher.")

# 4. Config-Flags für das Notebook explizit auf True setzen
# Damit erzwingen wir, dass on_validation_epoch_end die Zarrs und CSVs schreibt
model.cfg["testing"] = {
    "save_tensors": True,
    "save_metrics": True
}

# 5. Das große Stitching und Plotting manuell anstoßen!
print("🧵 Starte Stitching, Metric-Calculation und Zarr-Export...")
model.on_validation_epoch_end()

print("\n🎉 FERTIG! Schau in deinen run_dir/tensors und run_dir/metrics Ordner!")

In [ ]:
model.validation_step_outputs.clear()

In [ ]:
model.validation_step_outputs[0]["y_true"].shape